# Week 7-8 Report Notebook

Notebook riêng để truy vấn ChromaDB và sinh báo cáo so sánh + citation (tuần 7-8).

Yêu cầu: đã chạy xong `Ingestion_pipeline_test.ipynb` để có dữ liệu trong `chroma_db/.`

## Section 1: Import thư viện và kết nối ChromaDB

Import các thư viện cần thiết, xác định đường dẫn tới `chroma_db` và khởi tạo client/collection.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os
import json
import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CHROMA_DIR = ROOT / 'chroma_db'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CHROMA_DIR.exists(), (
    '[FAIL] Không tìm thấy thư mục ChromaDB tại: ' + str(CHROMA_DIR)
    + ' → Hãy chạy Ingestion_pipeline_test.ipynb trước!'
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection('legal_chunks')
total = collection.count()

print('[OK] Kết nối ChromaDB thành công')
print(f'     Collection : legal_chunks')
print(f'     Tổng chunks: {total}')
print(f'     Chroma path: {CHROMA_DIR}')

### Output mẫu

[OK] Kết nối ChromaDB thành công
     Collection : legal_chunks
     Tổng chunks: 1234
     Chroma path: c:\Users\user\rag-based-legal-comparison\chroma_db

## Section 2: Tải embedding model

Load model `SentenceTransformer` để mã hóa câu truy vấn trước khi tìm kiếm.

In [ ]:
load_dotenv(ROOT / '.env')
HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN or HF_TOKEN == 'hf_your_token_here':
    raise ValueError('Chưa cấu hình HF_TOKEN trong .env hoặc giá trị chưa hợp lệ.')

print('...Đang load embedding model...')
embedder = SentenceTransformer('huyydangg/DEk21_hcmute_embedding', token=HF_TOKEN)
print('[OK] Embedding model sẵn sàng')

## Section 3: Báo cáo tổng quan ChromaDB

Lấy dữ liệu từ collection, đếm phân bố theo `doc_id`, `chunk_type`, chương và độ dài nội dung.

In [ ]:
all_data = collection.get(include=['metadatas', 'documents'])
all_meta = all_data['metadatas']
all_docs = all_data['documents']

doc_counts = Counter(m['doc_id'] for m in all_meta)
type_counts = Counter(m.get('chunk_type', 'unknown') for m in all_meta)
chapter_counts = Counter(m.get('chuong_so', '0') if m.get('chuong_so', '0') != '0' else 'header' for m in all_meta)
char_lens = [len(d) for d in all_docs] if all_docs else [0]

SEP = '═' * 65
print(SEP)
print('BÁO CÁO TỔNG QUAN ChromaDB')
print(SEP)
print(f'  Tổng chunks      : {total}')
print(f'  Số văn bản luật  : {len(doc_counts)}')
print()
print('--- Phân bố theo doc_id ---')
for doc_id, cnt in sorted(doc_counts.items()):
    print(f'  {doc_id:<40} {cnt:>6}')
print()
print('--- Phân bố theo chunk_type ---')
for t, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t:<12} {cnt:>6}')
print()
print('--- Phân bố theo chương ---')
for label, cnt in sorted(chapter_counts.items(), key=lambda x: -x[1])[:20]:
    print(f'  {label:<10} {cnt:>6}')
print()
print('--- Thống kê độ dài chunk ---')
print(f'  Ngắn nhất : {min(char_lens)} ký tự')
print(f'  Dài nhất  : {max(char_lens)} ký tự')
print(f'  Trung bình: {sum(char_lens)//len(char_lens)} ký tự')
print(SEP)

## Section 4: Demo truy vấn ngữ nghĩa

Segment câu hỏi tiếng Việt, embed câu hỏi, truy vấn ChromaDB và hiển thị kết quả.

In [ ]:
def show_chunk(rank, chunk_id, meta, document, distance=None):
    SEP = '-' * 65
    score = f' | score={(1/(1+distance)):.4f}' if distance is not None else ''
    title = meta.get('article_title') or '(không có tiêu đề)'
    print(SEP)
    print(f'[{rank}] {meta.get("clause_id","unknown")} — {title}{score}')
    print(f'    doc_id      : {meta.get("doc_id","unknown")}')
    print(f'    chunk_type  : {meta.get("chunk_type","unknown")}')
    print(f'    location    : Chương {meta.get("chuong_so","?")} / Mục {meta.get("muc_so","?")}')
    print(f'    khoan/diem  : {meta.get("khoan_count","?")} / {meta.get("diem_count","?")}')
    print('    text:')
    for line in document.strip().split('\n')[:6]:
        print(f'      {line}')
    if len(document) > 400:
        print('      ...')

QUERY = 'điều kiện để được cấp phép xây dựng'
TOP_K = 5

query_seg = ViTokenizer.tokenize(QUERY)
query_vec = embedder.encode([query_seg], convert_to_numpy=True)
results = collection.query(query_embeddings=query_vec.tolist(), n_results=TOP_K, include=['metadatas', 'documents', 'distances'])

print('')
print(f'🔍 Truy vấn ngữ nghĩa: {QUERY}')
for rank, (cid, metas, docs, dists) in enumerate(zip(results['ids'][0], results['metadatas'][0], results['documents'][0], results['distances'][0]), start=1):
    show_chunk(rank, cid, metas, docs, distance=dists)

## Section 5: Demo lọc metadata

Lọc theo `article_number` hoặc `doc_id` để lấy chính xác các điều khoản.

In [ ]:
FILTER_ARTICLE_NUMBER = '1'  # Thay đổi số điều
FILTER_DOC_ID = None       # Hoặc đặt doc_id cụ thể

where_clause = {'article_number': FILTER_ARTICLE_NUMBER} if FILTER_DOC_ID is None else {'$and': [{'article_number': FILTER_ARTICLE_NUMBER}, {'doc_id': FILTER_DOC_ID}]}
result = collection.get(where=where_clause, include=['metadatas', 'documents'])

print('')
print('🔎 Lọc metadata')
print(f'  article_number = {FILTER_ARTICLE_NUMBER}')
print(f'  doc_id         = {FILTER_DOC_ID or '(tất cả)'}')
print(f'  Kết quả        = {len(result['ids'])}')
for idx, (cid, meta, doc) in enumerate(zip(result['ids'], result['metadatas'], result['documents']), start=1):
    show_chunk(idx, cid, meta, doc)

### Kết quả lọc metadata mẫu

🔍 Lọc metadata
  article_number = 1
  doc_id         = (tất cả)
  Kết quả        = 3

─────────────────────────────────────────────────────────────────
[1] Dieu_01 — Quy định chung về xây dựng
    doc_id      : 112_2025_QH15_586814
    chunk_type  : article
    location    : Chương I / Mục 1
    khoan/diem  : 2 / 0
    text:
      Điều 1. Các điều kiện để được cấp phép xây dựng gồm hồ sơ hợp lệ, thiết kế kỹ thuật, và bảo đảm an toàn.
─────────────────────────────────────────────────────────────────
[2] Dieu_02 — Thẩm quyền cấp phép
    doc_id      : 112_2025_QH15_586814
    chunk_type  : article
    location    : Chương II / Mục 1
    khoan/diem  : 1 / 0
    text:
      Điều 2. Cơ quan cấp phép xây dựng là ủy ban nhân dân cấp tỉnh đối với công trình dân dụng.


## Section 6: Demo hybrid search

Kết hợp tìm kiếm ngữ nghĩa với filter metadata để giới hạn trong văn bản hoặc chương cụ thể.

In [ ]:
HYBRID_QUERY = 'quyền hạn và trách nhiệm của cơ quan nhà nước'
HYBRID_DOC_ID = None
HYBRID_CHUONG = None
HYBRID_TOP_K = 4

hybrid_seg = ViTokenizer.tokenize(HYBRID_QUERY)
hybrid_vec = embedder.encode([hybrid_seg], convert_to_numpy=True)
query_kwargs = {'query_embeddings': hybrid_vec.tolist(), 'n_results': HYBRID_TOP_K, 'include': ['metadatas', 'documents', 'distances']}
filters = []
if HYBRID_DOC_ID:
    filters.append({'doc_id': HYBRID_DOC_ID})
if HYBRID_CHUONG:
    filters.append({'chuong_so': HYBRID_CHUONG})
if filters:
    query_kwargs['where'] = {'$and': filters} if len(filters) > 1 else filters[0]

results = collection.query(**query_kwargs)

print('')
print(f'🔍 Hybrid search: {HYBRID_QUERY}')
print(f'  doc_id filter = {HYBRID_DOC_ID or '(tất cả)'}')
print(f'  chuong filter = {HYBRID_CHUONG or '(tất cả)'}')
for rank, (cid, meta, doc, dist) in enumerate(zip(results['ids'][0], results['metadatas'][0], results['documents'][0], results['distances'][0]), start=1):
    show_chunk(rank, cid, meta, doc, distance=dist)

## Kết quả demo mẫu

🔍 HYBRID SEARCH
   Câu hỏi   : "quyền hạn và trách nhiệm của cơ quan nhà nước"
   Giới hạn  : 112_2025_QH15_586814
   → Top 3 kết quả:

─────────────────────────────────────────────────────────────────
  [1] Dieu_12 — Quản lý nhà nước về quy hoạch
       Loại     : article  |  Do lien quan: 0.0202
       Nguồn    : 112_2025_QH15_586814
       Vị trí   : Chương I  /  (Không có Mục)
       Khoản/Điểm: 3 khoản,  0 điểm
       Chunk ID : 112_2025_QH15_586814_v1_0012
       Nội dung :
         Điều 12. Quản lý nhà nước về quy hoạch
         1. Chính phủ thống nhất quản lý nhà nước về quy hoạch trong phạm vi cả nước.
         2. Bộ Tài chính là cơ quan đầu mối giúp Chính phủ thực hiện quản lý nhà nước về quy hoạch.
         3. Các Bộ, Ủy ban nhân dân các cấp thực hiện quản lý nhà nước về quy hoạch trong phạm vi nhiệm vụ, quyền hạn được giao.
─────────────────────────────────────────────────────────────────
  [2] Dieu_17 — Thẩm quyền tổ chức lập quy hoạch
       Loại     : article  |  Do lien quan: 0.0177
       Nguồn    : 112_2025_QH15_586814
       Vị trí   : Chương II  /  Mục 1
       Khoản/Điểm: 4 khoản,  0 điểm
       Chunk ID : 112_2025_QH15_586814_v1_0017
       ...
         1. Chính phủ tổ chức thực hiện quy hoạch tổng thể quốc gia, quy hoạch không gian biển quốc gia, quy hoạch sử dụng đất quốc gia; tổ chức triển khai chương trình, dự án để thực hiện quy hoạch tổng thể quốc gia, quy hoạch không gian biển quốc gia, quy hoạch sử dụng đất quốc gia.
         2. Thủ tướng Chính phủ tổ chức thực hiện quy hoạch vùng; tổ chức
         ... [656 ký tự tổng cộng]

## Section 7: So sánh hai phiên bản văn bản

So sánh hai `doc_id` khác nhau, phân loại `added / removed / modified` và chuẩn bị prompt cho LLM.

In [ ]:
doc_ids = sorted({m['doc_id'] for m in all_meta})
if len(doc_ids) < 2:
    raise ValueError('Cần ít nhất 2 doc_id khác nhau trong ChromaDB để so sánh.')
BASE_DOC_ID = doc_ids[0]
NEW_DOC_ID = doc_ids[1]
# BASE_DOC_ID = 'v1_doc_id_here'
# NEW_DOC_ID = 'v2_doc_id_here'

print(f'So sánh: {BASE_DOC_ID} → {NEW_DOC_ID}')

chunks_by_doc = defaultdict(dict)
for meta, doc in zip(all_meta, all_docs):
    chunks_by_doc[meta['doc_id']][meta['clause_id']] = {'meta': meta, 'text': doc}

base_map = chunks_by_doc[BASE_DOC_ID]
new_map = chunks_by_doc[NEW_DOC_ID]
all_clause_ids = sorted(set(base_map) | set(new_map))

added = []
removed = []
modified = []
unchanged = []

for clause_id in all_clause_ids:
    base_item = base_map.get(clause_id)
    new_item = new_map.get(clause_id)
    if base_item is None:
        added.append(new_item)
    elif new_item is None:
        removed.append(base_item)
    elif base_item['text'].strip() != new_item['text'].strip():
        modified.append({'clause_id': clause_id, 'base': base_item, 'new': new_item})
    else:
        unchanged.append(base_item)

print('')
print('=== Kết quả so sánh ===')
print(f'  Added    : {len(added)}')
print(f'  Removed  : {len(removed)}')
print(f'  Modified : {len(modified)}')
print(f'  Unchanged: {len(unchanged)}')

def clean_text_snippet(text, limit=240):
    normalized = ' '.join(text.split())
    return normalized[:limit] + ('...' if len(normalized) > limit else '')

prompt_lines = [
    'Bạn là trợ lý so sánh văn bản pháp lý tiếng Việt.',
    f'Base: {BASE_DOC_ID}',
    f'New : {NEW_DOC_ID}',
    '',
    'Yêu cầu:',
    '- Phân loại thay đổi thành: added / deleted / modified.',
    '- Mỗi mục phải kèm chứng cứ cụ thể.',
    '- Nếu không có bằng chứng rõ ràng, ghi rõ Không có bằng chứng và không kết luận.',
    '',
    '=== Output format ===',
    'Trả về JSON với: summary, added, removed, modified, citations.',
    ''
]
for item in added[:5]:
    prompt_lines.append(f"- Added: {item['meta']['clause_id']} | {item['meta'].get('article_title','')}")
    prompt_lines.append(f"  Evidence: {clean_text_snippet(item['text'])}")
    prompt_lines.append('')
for item in removed[:5]:
    prompt_lines.append(f"- Removed: {item['meta']['clause_id']} | {item['meta'].get('article_title','')}")
    prompt_lines.append(f"  Evidence: {clean_text_snippet(item['text'])}")
    prompt_lines.append('')
for item in modified[:5]:
    prompt_lines.append(f"- Modified: {item['clause_id']}")
    prompt_lines.append(f"  Base: {clean_text_snippet(item['base']['text'])}")
    prompt_lines.append(f"  New : {clean_text_snippet(item['new']['text'])}")
    prompt_lines.append('')

prompt_text = '\n'.join(prompt_lines)
prompt_path = OUTPUT_DIR / 'week7_8_prompt.txt'
prompt_path.write_text(prompt_text, encoding='utf-8')
print(f'\n[OK] Prompt đã lưu: {prompt_path}')

## Section 8: Tạo báo cáo thay đổi và citations

Xây dựng báo cáo JSON mẫu từ kết quả so sánh và in các mục `added` / `removed` / `modified` cùng `citations`.

In [ ]:
report = {
    'summary': f'So sánh {BASE_DOC_ID} → {NEW_DOC_ID}: {len(added)} added, {len(removed)} removed, {len(modified)} modified.',
    'added': [
        {
            'clause_id': item['meta']['clause_id'],
            'article_title': item['meta'].get('article_title',''),
            'note': 'Điều khoản mới thêm ở phiên bản mới.',
            'evidence': {
                'source': item['meta']['doc_id'],
                'article_title': item['meta'].get('article_title',''),
                'text_snippet': clean_text_snippet(item['text'])
            }
        }
        for item in added[:5]
    ],
    'removed': [
        {
            'clause_id': item['meta']['clause_id'],
            'article_title': item['meta'].get('article_title',''),
            'note': 'Điều khoản bị xóa ở phiên bản mới.',
            'evidence': {
                'source': item['meta']['doc_id'],
                'article_title': item['meta'].get('article_title',''),
                'text_snippet': clean_text_snippet(item['text'])
            }
        }
        for item in removed[:5]
    ],
    'modified': [
        {
            'clause_id': item['clause_id'],
            'article_title': item['base']['meta'].get('article_title',''),
            'note': 'Nội dung điều khoản thay đổi.',
            'base_evidence': {
                'source': item['base']['meta']['doc_id'],
                'text_snippet': clean_text_snippet(item['base']['text'])
            },
            'new_evidence': {
                'source': item['new']['meta']['doc_id'],
                'text_snippet': clean_text_snippet(item['new']['text'])
            }
        }
        for item in modified[:5]
    ],
    'citations': []
}
for item in added[:3] + removed[:3]:
    report['citations'].append({
        'source': item['meta']['doc_id'],
        'clause_id': item['meta']['clause_id'],
        'article_title': item['meta'].get('article_title',''),
        'text_snippet': clean_text_snippet(item['text'])
    })
for item in modified[:3]:
    report['citations'].append({
        'source': item['base']['meta']['doc_id'],
        'clause_id': item['clause_id'],
        'article_title': item['base']['meta'].get('article_title',''),
        'text_snippet': clean_text_snippet(item['base']['text'])
    })
    report['citations'].append({
        'source': item['new']['meta']['doc_id'],
        'clause_id': item['clause_id'],
        'article_title': item['new']['meta'].get('article_title',''),
        'text_snippet': clean_text_snippet(item['new']['text'])
    })

print('')
print('=== Báo cáo JSON mẫu ===')
print(json.dumps(report, ensure_ascii=False, indent=2))

## Output cuối cùng mẫu

[OK] Báo cáo JSON mẫu đã tạo xong.

- summary: So sánh `112_2025_QH15_586814` → `112_2025_QH15_586814_v2`: 12 added, 8 removed, 5 modified.
- added: 12
- removed: 8
- modified: 5
- citations: 8

```json
{
  "summary": "So sánh 112_2025_QH15_586814 → 112_2025_QH15_586814_v2: 12 added, 8 removed, 5 modified.",
  "added": [
    {
      "clause_id": "Dieu_03",
      "article_title": "Quy định chung về xây dựng",
      "note": "Điều khoản mới thêm ở phiên bản mới.",
      "evidence": {
        "source": "112_2025_QH15_586814_v2",
        "article_title": "Quy định chung về xây dựng",
        "text_snippet": "Các điều kiện cấp phép xây dựng được mở rộng cho cả dự án dân dụng và công nghiệp..."
      }
    }
  ],
  "removed": [
    {
      "clause_id": "Dieu_15",
      "article_title": "Quy định về hồ sơ",
      "note": "Điều khoản bị xóa ở phiên bản mới.",
      "evidence": {
        "source": "112_2025_QH15_586814",
        "article_title": "Quy định về hồ sơ",
        "text_snippet": "Hồ sơ đề nghị cấp giấy phép xây dựng gồm bản vẽ thiết kế, giấy tờ pháp lý..."
      }
    }
  ],
  "modified": [
    {
      "clause_id": "Dieu_21",
      "article_title": "Bảo đảm an toàn",
      "note": "Nội dung điều khoản thay đổi.",
      "base_evidence": {
        "source": "112_2025_QH15_586814",
        "text_snippet": "Công trình phải đảm bảo an toàn cho người và tài sản trong phạm vi công trình..."
      },
      "new_evidence": {
        "source": "112_2025_QH15_586814_v2",
        "text_snippet": "Công trình phải đảm bảo an toàn cho người, tài sản và môi trường xung quanh..."
      }
    }
  ],
  "citations": [
    {
      "source": "112_2025_QH15_586814_v2",
      "clause_id": "Dieu_03",
      "article_title": "Quy định chung về xây dựng",
      "text_snippet": "Các điều kiện cấp phép xây dựng được mở rộng cho cả dự án dân dụng và công nghiệp..."
    }
  ]
}
```